# Topic embeddings and comment means in one t-SNE projection

Circles show SGNS topic embeddings and diamonds show mean comment embeddings. A thin line joins the two representations of the same full topic path. Unpaired topics appear as individual points.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import qualitative
from scipy.linalg import orthogonal_procrustes
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "topic_embeddings").is_dir())
TOPIC_KEY = ["Topic_1", "Topic_2", "Topic_3"]
SEED = 42
PERPLEXITY = 20

topics = pd.read_csv(ROOT / "results/sgns/topic_embeddings_normalized.csv", keep_default_na=False)
means = pd.read_csv(ROOT / "data/processed/comment_embedding_means.csv", keep_default_na=False)
pairs = topics[TOPIC_KEY].reset_index(names="topic_row").merge(
    means[TOPIC_KEY].reset_index(names="mean_row"), on=TOPIC_KEY, validate="one_to_one"
)
print(f"{len(topics):,} topic embeddings, {len(means):,} comment means, {len(pairs):,} matched paths")

1,000 topic embeddings, 600 comment means, 417 matched paths


## Align on shared topic paths

1. **Normalize each vector.** Divide each topic embedding and comment mean by its own length so each starts with length 1.
2. **Center the topics and reduce the comment means.** Subtract the average matched topic vector from every topic vector, keeping all 150 dimensions. Fit PCA on the matched comment means and use it to project all comment means from 896 to 150 dimensions.
3. **Put the spaces on the same scale.** In each space, compute the root mean square (RMS) length of its matched vectors. Divide every vector in that space by this single number so the matched vectors have an RMS length of 1.
4. **Align the comment means to the topics.** Use the 417 matched paths to fit one orthogonal transformation (a rotation or reflection) that minimizes the sum of squared distances between paired vectors. Apply it to every comment mean.

In [ ]:
topic_vectors = topics.filter(regex=r"^embedding_\d+$").to_numpy(dtype=float)
mean_vectors = means.filter(regex=r"^dim_\d+$").to_numpy(dtype=float)
topic_rows = pairs["topic_row"].to_numpy()
mean_rows = pairs["mean_row"].to_numpy()
dims = topic_vectors.shape[1]
if dims > min(len(pairs) - 1, mean_vectors.shape[1]):
    raise ValueError(f"Too few matched paths or comment dimensions to fit {dims} PCA components.")

topic_coords = normalize(topic_vectors)
topic_coords -= topic_coords[topic_rows].mean(axis=0)

unit_means = normalize(mean_vectors)
mean_pca = PCA(n_components=dims, svd_solver="full").fit(unit_means[mean_rows])
mean_coords = mean_pca.transform(unit_means)

def scale_to_matched_rms(coords, rows):
    """Scale coordinates to have the same root mean square as the matched vectors."""
    rms = np.sqrt(np.square(coords[rows]).sum(axis=1).mean())
    if rms == 0:
        raise ValueError("Matched vectors have no variation.")
    return coords / rms

topic_coords = scale_to_matched_rms(topic_coords, topic_rows)
mean_coords = scale_to_matched_rms(mean_coords, mean_rows)
rotation, _ = orthogonal_procrustes(mean_coords[mean_rows], topic_coords[topic_rows])
aligned_means = mean_coords @ rotation

combined = np.vstack([topic_coords, aligned_means])
xy = TSNE(n_components=2, perplexity=PERPLEXITY, init="pca", learning_rate="auto",
          max_iter=1000, random_state=SEED).fit_transform(combined)

## Explore the projection

Hover over a point for its path and source. Click a main topic in the legend to toggle both sets of points and their connecting lines.

In [3]:
sources = {"Topic embedding": "circle", "Comment mean": "diamond"}
points = pd.concat(
    [topics[TOPIC_KEY].assign(source="Topic embedding"), means[TOPIC_KEY].assign(source="Comment mean")],
    ignore_index=True,
)
points[["x", "y"]] = xy
points["path"] = points[TOPIC_KEY].apply(lambda row: " / ".join(filter(None, row)), axis=1)

fig = go.Figure()
for main_topic, group in pairs.groupby("Topic_1", sort=False):
    starts = xy[group["topic_row"]]
    ends = xy[len(topics) + group["mean_row"]]
    # NaN separators keep adjacent pairs from being joined.
    segments = np.stack([starts, ends, np.full_like(starts, np.nan)], axis=1).reshape(-1, 2)
    fig.add_scatter(x=segments[:, 0], y=segments[:, 1], mode="lines",
                    line=dict(color="rgba(75,75,75,0.20)", width=0.6),
                    hoverinfo="skip", legendgroup=main_topic, showlegend=False)

palette = qualitative.Plotly + qualitative.Dark24
for i, (main_topic, group) in enumerate(points.groupby("Topic_1", sort=False)):
    color = palette[i % len(palette)]
    for j, (source, rows) in enumerate(group.groupby("source", sort=False)):
        fig.add_scatter(
            x=rows.x, y=rows.y, mode="markers", name=main_topic, legendgroup=main_topic,
            showlegend=(j == 0), marker=dict(symbol=sources[source], color=color, size=7, opacity=0.8),
            text=rows.path, hovertemplate="%{text}<br>" + source + "<extra></extra>",
        )

fig.update_layout(
    title="Topic embeddings (circles) and comment means (diamonds)",
    xaxis_title="t-SNE 1", yaxis_title="t-SNE 2", legend_title_text="Main topic",
    legend_groupclick="togglegroup", template="plotly_white", height=800,
)
output = ROOT / "output/topic_comment_means_joint_tsne.html"
output.parent.mkdir(exist_ok=True)
fig.write_html(output)
fig.show()

In [4]:
# Top 10 neighbors of lifestyle / raetsel-sudoku / sudoku
from IPython.display import display

sudoku_path = ("lifestyle", "raetsel-sudoku", "sudoku")
match = pairs.set_index(TOPIC_KEY).loc[sudoku_path]
topic_idx, mean_idx = int(match["topic_row"]), int(match["mean_row"])
is_sudoku = points[TOPIC_KEY].eq(sudoku_path, axis="columns").all(axis=1).to_numpy(copy=True)

def show_neighbors(title, candidates, distances, exclude):
    table = candidates[["path", "source"]].assign(distance=distances)
    table = table.loc[~exclude].nsmallest(10, "distance").reset_index(drop=True)
    table.index = pd.RangeIndex(1, len(table) + 1, name="Rank")
    print(title)
    display(table.round({"distance": 4}))

# Rank by cosine distance in each original space. Smaller values mean closer neighbors.
for title, vectors, query_idx, rows in [
    ("Topic space · cosine distance", topic_vectors, topic_idx, slice(0, len(topics))),
    ("Comment mean space · cosine distance", mean_vectors, mean_idx, slice(len(topics), None)),
]:
    unit = normalize(vectors)
    distances = np.clip(1 - unit @ unit[query_idx], 0, 2)
    show_neighbors(title, points.iloc[rows], distances, is_sudoku[rows])

# Query both Sudoku points against all aligned vectors using the joint t-SNE input metric.
for source, query_idx in [("topic embedding", topic_idx), ("comment mean", len(topics) + mean_idx)]:
    distances = np.linalg.norm(combined - combined[query_idx], axis=1)
    show_neighbors(f"Shared space from Sudoku {source} · Euclidean distance",
                   points, distances, is_sudoku)

Topic space · cosine distance


,path,source,distance
Rank,,,
1,diskurs / blogs / quizblog,Topic embedding,0.5333
2,diskurs / offtopic,Topic embedding,0.5514
3,web / games / gewinnspielegames,Topic embedding,0.5718
4,diskurs / userblogs / quizblog,Topic embedding,0.5767
5,sport / fussball / werderbremen,Topic embedding,0.5891
6,zukunft / ideen-fuer-eine-bessere-welt,Topic embedding,0.5950
7,web / webmix / webquiz,Topic embedding,0.5954
8,kultur / literatur / literaturforumuser,Topic embedding,0.6046
9,panorama / oesterreich / laenderchronik,Topic embedding,0.6083


Comment mean space · cosine distance


,path,source,distance
Rank,,,
1,lifestyle / raetsel-sudoku,Comment mean,0.0000
2,sport / tennis / daviscup,Comment mean,0.1142
3,sport / wintersport / skisprung,Comment mean,0.1201
4,sport / wintersport / streif,Comment mean,0.1213
5,kultur / photoblognewald,Comment mean,0.1216
6,sport / wintersport,Comment mean,0.1232
7,sport / fussball / austriawien,Comment mean,0.1232
8,sport / wintersport / damenweltcup,Comment mean,0.1234
9,sport / wintersport / eishockeyliga,Comment mean,0.1237


Shared space from Sudoku topic embedding · Euclidean distance


,path,source,distance
Rank,,,
1,lifestyle / raetsel-sudoku,Comment mean,1.0432
2,diskurs / cartoons,Comment mean,1.0619
3,lifestyle,Comment mean,1.0775
4,etat,Comment mean,1.0994
5,panorama,Comment mean,1.1007
6,inland / serienundblogs,Comment mean,1.1036
7,inland / serienundblogs / cremersphotoblog,Comment mean,1.1036
8,web / webmix,Comment mean,1.1049
9,etat / blogs,Comment mean,1.1113


Shared space from Sudoku comment mean · Euclidean distance


,path,source,distance
Rank,,,
1,lifestyle / raetsel-sudoku,Comment mean,0.0000
2,sport / tennis / daviscup,Comment mean,1.3893
3,sport / wintersport / skisprung,Comment mean,1.4371
4,kultur / photoblognewald,Comment mean,1.4441
5,sport / wintersport / streif,Comment mean,1.4523
6,kultur / musik / kritiken,Comment mean,1.4553
7,sport / wintersport / damenweltcup,Comment mean,1.4572
8,kultur / film / ansichtssachen-filmstarts,Comment mean,1.4587
9,sport / wintersport / eishockeyliga,Comment mean,1.4588
